In [1]:
import os
import sys
import argparse
import shutil
import random
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, datasets
from PIL import Image
import numpy as np

In [15]:
# CATEGORIES = ["chair", "table", "sofa", "bed", "cabinet", 
#               "bookshelf", "desk", "bench", "swivelchair", "wardrobe"]

In [16]:
CATEGORIES = ["bed", "chair", "sofa", "swivelchair", "table"]

In [3]:
DATA_DIR = Path("classifier_data")

In [ ]:
# for split in ["train", "val"]:
#     for cat in CATEGORIES:
#         folder = DATA_DIR / split / cat
#         folder.mkdir(parents=True, exist_ok=True)
        
# print("Folders created!")
# print(f"Location: {DATA_DIR.resolve()}")

Folders created!
Location: D:\Tripo New\classifier_data


In [ ]:
# for split in ["train", "val"]:
#     folder = DATA_DIR / split / "swivelchair"
#     folder.mkdir(parents=True, exist_ok=True)
# print("Folders created!")
# print(f"Location: {DATA_DIR.resolve()}")

Folders created!
Location: D:\Tripo New\classifier_data


In [22]:
class FurnitureClassifier(nn.Module):
    def __init__(self,num_classes=10, backbone_name="dinov2_vits14"):
        super().__init__()
        self.backbone = torch.hub.load("facebookresearch/dinov2", backbone_name,pretrained=True)
        self.backbone.eval()
        for param in self.backbone.parameters():
            param.requires_grad = False
        self.head = nn.Sequential(
            nn.Linear(384 ,256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256,128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128,num_classes)
        )
    def forward(self,images):
        with torch.no_grad():
            features = self.backbone(images)
        logits = self.head(features)
        return logits

In [8]:
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

In [9]:
IMAGE_SIZE = 224
BATCH_SIZE = 16

In [12]:
train_transform = transforms.Compose([
     transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),
     transforms.RandomHorizontalFlip(p=0.5),
     transforms.ColorJitter(brightness=0.2,contrast=0.2),
     transforms.ToTensor(),
     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [13]:
val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225]),
])

In [18]:
train_dataset = datasets.ImageFolder("classifier_data/train", transform=train_transform)
val_dataset = datasets.ImageFolder("classifier_data/val", transform=val_transform)


In [19]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training images: {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)}")
print(f"Classes: {train_dataset.classes}")

Training images: 4024
Validation images: 423
Classes: ['bed', 'chair', 'sofa', 'swivelchair', 'table']


In [20]:
device = "cuda" if torch.cuda.is_available() else "cpu"
num_classes = len(train_dataset.classes)

In [23]:
model = FurnitureClassifier (num_classes = num_classes)
model= model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.head.parameters(), lr=1e-3)

Using cache found in C:\Users\Ziads/.cache\torch\hub\facebookresearch_dinov2_main


In [25]:
NUM_EPOCHS = 15
best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):
    # Train
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        predictions = model(images)
        loss = criterion(predictions, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = predictions.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()
    
    # Validate
    model.eval()
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            predictions = model(images)
            _, predicted = predictions.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    
    train_acc = 100 * train_correct / train_total
    val_acc = 100 * val_correct / val_total
    
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Train Acc: {train_acc:.1f}% | "
          f"Val Acc: {val_acc:.1f}%"
          + (" ← best!" if val_acc > best_val_acc else ""))
    os.makedirs("classifier", exist_ok=True)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'model_state_dict': model.state_dict(),
            'classes': train_dataset.classes,
            'num_classes': num_classes,
        }, "classifier/furniture_classifier.pth")

print(f"\nDone! Best accuracy: {best_val_acc:.1f}%")

Epoch 1/15 | Train Acc: 99.0% | Val Acc: 97.9% ← best!
Epoch 2/15 | Train Acc: 99.2% | Val Acc: 99.1% ← best!
Epoch 3/15 | Train Acc: 99.2% | Val Acc: 98.8%
Epoch 4/15 | Train Acc: 99.1% | Val Acc: 99.5% ← best!
Epoch 5/15 | Train Acc: 99.6% | Val Acc: 98.6%
Epoch 6/15 | Train Acc: 99.2% | Val Acc: 99.3%
Epoch 7/15 | Train Acc: 99.5% | Val Acc: 99.3%
Epoch 8/15 | Train Acc: 99.7% | Val Acc: 99.5%
Epoch 9/15 | Train Acc: 99.7% | Val Acc: 99.3%
Epoch 10/15 | Train Acc: 99.8% | Val Acc: 99.5%
Epoch 11/15 | Train Acc: 99.7% | Val Acc: 99.5%
Epoch 12/15 | Train Acc: 99.5% | Val Acc: 99.1%
Epoch 13/15 | Train Acc: 99.3% | Val Acc: 99.1%
Epoch 14/15 | Train Acc: 99.6% | Val Acc: 99.3%
Epoch 15/15 | Train Acc: 99.7% | Val Acc: 99.5%

Done! Best accuracy: 99.5%


In [31]:
from PIL import Image

def predict(image_path):
    image = Image.open(image_path).convert("RGB")
    input_tensor = val_transform(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        logits = model(input_tensor)
        probs = torch.softmax(logits, dim=1)[0]
    
    top3_probs, top3_idx = probs.topk(3)
    print(f"\n{image_path}:")
    for i in range(3):
        name = train_dataset.classes[top3_idx[i]]
        conf = top3_probs[i].item() * 100
        print(f"  {name}: {conf:.1f}%")

predict("Chairs/example5.png")


Chairs/example5.png:
  chair: 53.7%
  swivelchair: 44.3%
  sofa: 1.7%
